In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import wandb
wandb.login(key="25a61b596081a070a75d3cfc16bfad96632650fc")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: wgierulski (wut-wgierulski). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/jupyter/.netrc


True

In [3]:
config={
"base_learning_rate": 0.0001,
"label_smoothing": 0.1,
"epochs": 10,
"layers": 12,
"frames": 8,
"val_epochs": 2,
"batch_size": 16
}

In [4]:
wandb.init(
    project="msc",
    config=config
)

In [5]:
wandb.define_metric("epoch_nr")
wandb.define_metric("train_accuracy_step")
wandb.define_metric("train_accuracy_epoch", step_metric="epoch_nr")
wandb.define_metric("valid_accuracy_epoch", step_metric="epoch_nr")
wandb.define_metric("valid_loss_epoch", step_metric="epoch_nr")
wandb.define_metric("train_loss_step")
wandb.define_metric("train_loss_epoch", step_metric="epoch_nr")

In [6]:
from torchvision.datasets import HMDB51
import torch
import matplotlib.pyplot as plt
import tqdm

In [7]:
device = "cuda"

In [8]:
def show_img(image):
    plt.imshow(image.permute(1,2,0).detach().cpu().numpy()*0.2+0.42)
    plt.show()

In [13]:
from transforms import Div255
from torchvision.transforms.v2 import ScaleJitter, RandAugment, RandomCrop, Compose, ColorJitter, RandomHorizontalFlip, Normalize, RandomApply, UniformTemporalSubsample, CenterCrop

ds_transform_train = Compose(transforms=[
    UniformTemporalSubsample(num_samples=config["frames"]),
    RandomHorizontalFlip(p=0.5),
    ScaleJitter(target_size=(240, 320), scale_range=(0.9, 1.33)),
    RandomCrop(size=(224,224), pad_if_needed=True, fill=128),
    RandomApply(p=0.8, transforms=[ColorJitter(brightness=0.2, contrast=0.1, saturation=0.1, hue=0.1)]),
    RandAugment(num_ops=2, magnitude=10, fill=128),
    Div255(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

ds_transform_valid = Compose(transforms=[
    UniformTemporalSubsample(num_samples=config["frames"]),
    CenterCrop((224, 224)),
    Div255(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

In [14]:
dataset_train = HMDB51(root="datasets/hmdb/data",
                 annotation_path="datasets/hmdb/test_train_splits/testTrainMulti_7030_splits",
                 frames_per_clip=60,
                 step_between_clips=30,
                 num_workers=4,
                 transform=ds_transform_train,
                 train=True,
                 output_format="TCHW")

  0%|          | 0/422 [00:00<?, ?it/s]

100%|██████████| 422/422 [00:55<00:00,  7.61it/s]
/opt/conda/lib/python3.10/site-packages/torchvision/datasets/video_utils.py:219: UserWarning: There aren't enough frames in the current video to get a clip for the given clip length and frames between clips. The video (and potentially others) will be skipped.
  warnings.warn(


In [15]:
dataset_test = HMDB51(root="datasets/hmdb/data",
                 annotation_path="datasets/hmdb/test_train_splits/testTrainMulti_7030_splits",
                 frames_per_clip=60,
                 step_between_clips=30,
                 num_workers=4,
                 transform=ds_transform_valid,
                 train=False,
                 output_format="TCHW")

  0%|          | 0/422 [00:00<?, ?it/s]

100%|██████████| 422/422 [00:55<00:00,  7.57it/s]


In [16]:
print(dataset_train)
print(dataset_test)

Dataset HMDB51
    Number of datapoints: 6076
    Root location: datasets/hmdb/data
Dataset HMDB51
    Number of datapoints: 2436
    Root location: datasets/hmdb/data


In [ ]:
from pytorchvideo.data.hmdb51 import Hmdb51
from pytorchvideo.data.clip_sampling import UniformClipSampler

dataset_test = Hmdb51(data_path=r"datasets\hmdb\test_train_splits\testTrainMulti_7030_splits",
                      clip_sampler=UniformClipSampler(1000),
                      decode_audio=False,
                      video_path_prefix="datasets/hmdb/data",
                      split_type="test")

In [17]:
from transformers import ViTForImageClassification
vit = ViTForImageClassification.from_pretrained("WinKawaks/vit-small-patch16-224")

In [18]:
from transformers import TimesformerConfig, TimesformerForVideoClassification
timesformer_config = TimesformerConfig(num_frames=config["frames"], hidden_size=384, num_attention_heads=6, intermediate_size=384*4, num_labels=51, num_hidden_layers=config["layers"])
timesformer = TimesformerForVideoClassification(timesformer_config)
timesformer.to(device)

TimesformerForVideoClassification(
  (timesformer): TimesformerModel(
    (embeddings): TimesformerEmbeddings(
      (patch_embeddings): TimesformerPatchEmbeddings(
        (projection): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (time_drop): Dropout(p=0.0, inplace=False)
    )
    (encoder): TimesformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x TimesformerLayer(
          (drop_path): Identity()
          (attention): TimeSformerAttention(
            (attention): TimesformerSelfAttention(
              (qkv): Linear(in_features=384, out_features=1152, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
            )
            (output): TimesformerSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): TimesformerIntermediate(
            (dense

In [19]:
with torch.no_grad():
    timesformer.timesformer.embeddings.patch_embeddings.projection.load_state_dict(vit.vit.embeddings.patch_embeddings.projection.state_dict())
    timesformer.timesformer.embeddings.position_embeddings.copy_(vit.vit.embeddings.position_embeddings)
    #timesformer.timesformer.embeddings.time_embeddings
    for i in range(config["layers"]):
        vit_qkv_weight = torch.cat([vit.vit.encoder.layer[i].attention.attention.query.weight, vit.vit.encoder.layer[i].attention.attention.key.weight, vit.vit.encoder.layer[i].attention.attention.value.weight])
        vit_qkv_bias = torch.cat([vit.vit.encoder.layer[i].attention.attention.query.bias, vit.vit.encoder.layer[i].attention.attention.key.bias, vit.vit.encoder.layer[i].attention.attention.value.bias])

        timesformer.timesformer.encoder.layer[i].attention.attention.qkv.weight.copy_(vit_qkv_weight)
        timesformer.timesformer.encoder.layer[i].attention.attention.qkv.bias.copy_(vit_qkv_bias)
        timesformer.timesformer.encoder.layer[i].attention.output.dense.load_state_dict(vit.vit.encoder.layer[i].attention.output.dense.state_dict())
        timesformer.timesformer.encoder.layer[i].intermediate.dense.load_state_dict(vit.vit.encoder.layer[i].intermediate.dense.state_dict())
        timesformer.timesformer.encoder.layer[i].output.dense.load_state_dict(vit.vit.encoder.layer[i].output.dense.state_dict())
        timesformer.timesformer.encoder.layer[i].layernorm_before.load_state_dict(vit.vit.encoder.layer[i].layernorm_before.state_dict())
        timesformer.timesformer.encoder.layer[i].layernorm_after.load_state_dict(vit.vit.encoder.layer[i].layernorm_after.state_dict())
        timesformer.timesformer.encoder.layer[i].temporal_layernorm.load_state_dict(vit.vit.encoder.layer[i].layernorm_before.state_dict())
        timesformer.timesformer.encoder.layer[i].temporal_attention.attention.qkv.weight.copy_(vit_qkv_weight)
        timesformer.timesformer.encoder.layer[i].temporal_attention.attention.qkv.bias.copy_(vit_qkv_bias)
        timesformer.timesformer.encoder.layer[i].temporal_attention.output.dense.load_state_dict(vit.vit.encoder.layer[i].attention.output.dense.state_dict())
        #timesformer.timesformer.encoder.layer[i].temporal_dense
    timesformer.timesformer.layernorm.load_state_dict(vit.vit.layernorm.state_dict())

In [20]:
from torch.utils.data import DataLoader

training_loader = DataLoader(dataset=dataset_train, batch_size=config["batch_size"], shuffle=True)
validation_loader = DataLoader(dataset=dataset_test, batch_size=config["batch_size"], shuffle=False)

In [21]:
from models.vivit_custom.utils.scheduler import WarmupCosineSchedule
epochs = config["epochs"]
label_smoothing = config["label_smoothing"]

optimizer = torch.optim.SGD(timesformer.parameters(), lr=config["base_learning_rate"], momentum=0.9)
#scheduler = WarmupCosineSchedule(optimizer, warmup_steps=int(0.06 * len(training_loader)), t_total=len(training_loader) * epochs)
loss = torch.nn.functional.cross_entropy

In [26]:
from torchvision.transforms.v2 import MixUp
mixup = MixUp(alpha=0.1, num_classes=51).to(device)

In [ ]:
timesformer.train()


for epoch in range(epochs):
    print(f"Current epoch: {epoch+1}")
    curr_epoch_losses_train = []
    curr_epoch_accuracies_valid = []
    total_epoch_samples = 0
    total_epoch_correct_train = 0
    for i, data in enumerate(tqdm.tqdm(training_loader)):
        images, _, labels = data
        images = images.to(device)
        labels = labels.to(device)

        #images, labels_mixed_up = mixup(images, labels)

        optimizer.zero_grad()

        outputs = timesformer(images).logits

        loss_train = loss(outputs, labels, label_smoothing=label_smoothing)
        loss_train.backward()

        # logging
        _, predicted = torch.max(outputs, dim=1)
        total_epoch_samples += len(images)
        correct = (predicted == labels).sum().item()
        total_epoch_correct_train += correct
        step_accuracy = correct / len(images)
        curr_epoch_losses_train.append(loss_train.item())
        wandb.log({"train_accuracy_step": step_accuracy, "train_loss_step": loss_train.item()})

        #scheduler.step()
        optimizer.step()
    
    # logging
    wandb.log({
        "train_accuracy_epoch": total_epoch_correct_train / total_epoch_samples,
        "train_loss_epoch": sum(curr_epoch_losses_train) / len(training_loader),
        "epoch_nr": epoch + 1
               })
    
    if epoch % config["val_epochs"] == 0:
        total_loss = 0
        total_batches = 0
        total_correct = 0
        total_samples = 0
        with torch.no_grad():
            for i, data in enumerate(tqdm.tqdm(validation_loader)):
                images, _, labels = data
                images = images.to(device)
                labels = labels.to(device)

                outputs = timesformer(images).logits

                loss_valid = loss(outputs, labels)

                # logging
                _, predicted = torch.max(outputs, dim=1)
                correct = (predicted == labels).sum().item()
                total_batches += 1
                total_loss += loss_valid.item()
                total_correct += correct
                total_samples += len(images)

        wandb.log({"valid_accuracy_epoch": total_correct / total_samples,
                   "valid_loss_epoch": total_loss / total_batches
                   }
                   )



In [29]:
torch.cuda.is_available()

True

In [30]:
a = torch.Tensor([1,2]).to("cuda")
b = torch.Tensor([1,2]).to("cuda")

In [31]:
a*b

tensor([1., 4.], device='cuda:0')